[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

*Core Ops & Layers*
Implement **Layer Normalization** as a pure function.

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last** axis only.

### Signature
```python
def my_layer_norm(x, gamma, beta, eps=1e-5):
    ...
```

### Rules
- Do **not** use `nnx.LayerNorm` or `jax.nn.standardize`
- Normalise over the **last** axis
- Use the **biased** variance (`ddof=0`) — this is what every framework does
- `gamma` / `beta` are plain arrays of shape `(D,)`, passed in — not module state

### The two traps
1. **Unbiased variance.** `jnp.var(x, ddof=1)` divides by `n-1` and will not match
   any reference implementation. The population variance is correct here.
2. **Wrong axis.** Normalising over the batch axis is BatchNorm, not LayerNorm.
   LayerNorm is per-example, which is exactly why it works with batch size 1 and
   why transformers use it.

### Why a pure function
PyTorch would wrap $\gamma,\beta$ in a module. Here they are just arguments, so
the whole thing is a pure function of its inputs — `jit`, `grad` and `vmap` all
apply directly with nothing to thread through. This is the JAX default; reach
for `nnx.Module` only when you actually want the parameters to live somewhere.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def my_layer_norm(x, gamma, beta, eps=1e-5):
    """Layer normalization over the last axis.

    Args:
        x:     (..., D) array
        gamma: (D,) scale
        beta:  (D,) shift
        eps:   numerical-stability term inside the sqrt

    Returns:
        Array of the same shape as x.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

x = jax.random.normal(jax.random.key(0), (2, 3, 8)) * 5.0 + 2.0
gamma = jnp.ones(8)
beta = jnp.zeros(8)

out = my_layer_norm(x, gamma, beta)
print("shape:", out.shape)
print("per-row mean:", jnp.mean(out, axis=-1).ravel()[:4], "(~0)")
print("per-row std: ", jnp.std(out, axis=-1).ravel()[:4], "(~1)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("layernorm")

# hint("layernorm")      # stuck? nudge without the answer
# solution("layernorm")  # spoiler: the reference implementation